🎙️ Transcription & Diarisation avec WhisperX (Français)

Ce (très beau très magnifique) projet propose une solution complète pour transcrire des fichiers audio en français et identifier automatiquement les différents locuteurs (Diarization). Il utilise WhisperX, une version optimisée de Whisper (OpenAI) qui permet un alignement temporel précis et une séparation des voix via Pyannote. Sachez que j'ai pleuré quand ça a compilé.

✨ Fonctionnalités

    Transcription rapide : Utilise faster-whisper en arrière-plan.

    Alignement précis : Recalage des mots au millième de seconde près pour des timestamps parfaits.

    Diarisation : Identification des locuteurs (Speaker 1, Speaker 2, etc.).

    Correction automatique : Incorpore un patch de sécurité pour éviter les erreurs de récursion (RecursionError) liées aux versions récentes de Python et Torch.

🛠️ Prérequis
1. Accès aux modèles Pyannote

Pour utiliser la séparation des voix, vous devez impérativement :

    Avoir un compte sur Hugging Face.

    Accepter les conditions d'utilisation du modèle de diarisation : pyannote/speaker-diarization-3.1.

    Accepter les conditions du modèle de segmentation : pyannote/segmentation-3.0.

2. Token Hugging Face

Générez un token d'accès (type "Read") dans vos paramètres Hugging Face pour permettre au script de télécharger les modèles.
🚀 Installation

Il est recommandé d'utiliser un environnement avec GPU (NVIDIA). Sur Google Colab, choisissez l'accélérateur T4 GPU.
Bash

pip install whisperx huggingface_hub
pip install git+https://github.com/m-bain/whisperX.git --upgrade

📖 Utilisation

    Placez votre fichier audio (ex: audio.mp3) dans votre répertoire de travail.

    Dans le script, remplacez la variable YOUR_HF_TOKEN par votre jeton personnel.

    Exécutez le script pour obtenir la transcription segmentée par locuteur.

⚠️ Notes importantes sur le code

Le script inclut un Correctif de Sécurité (Patch) au début du code. Ce patch est essentiel pour :

    Éviter les boucles infinies lors de l'appel à la fonction get_prompt.

    Assurer la compatibilité entre les versions récentes de faster-whisper et l'architecture de whisperx.

⚖️ Licence

Ce projet est sous licence MIT. N'hésitez pas à l'utiliser et à le modifier pour vos propres besoins.

In [ ]:
!pip install whisperx
!pip install git+https://github.com/m-bain/whisperX.git --upgrade
# Note : On installe aussi huggingface_hub pour la connexion
!pip install huggingface_hub

In [ ]:
import whisperx
import torch
import whisperx.asr
from faster_whisper import WhisperModel
from huggingface_hub import login

# --- CONFIGURATION ---
device = "cuda" if torch.cuda.is_available() else "cpu"
audio_file = "ton_fichier.mp3" # À adapter
batch_size = 16
compute_type = "float16" # ou "int8" si pas de GPU performant
YOUR_HF_TOKEN = "VOTRE_TOKEN_ICI"

# --- CONNEXION ---
login(token=YOUR_HF_TOKEN)

# --- CORRECTIFS DE COMPATIBILITÉ (Très important pour GitHub) ---
# Ce patch évite les erreurs de récursion et les problèmes de versions de WhisperX/Faster-Whisper
if not hasattr(WhisperModel, "_is_patched"):
    class SafeOptions:
        def __init__(self, **kwargs): self.__dict__.update(kwargs)
        def __getattr__(self, name): return None
    whisperx.asr.TranscriptionOptions = SafeOptions
    original_get_prompt = WhisperModel.get_prompt
    def patched_get_prompt(self, tokenizer, previous_tokens, **kwargs):
        kwargs.pop('hotwords', None)
        return original_get_prompt(self, tokenizer, previous_tokens, **kwargs)
    WhisperModel.get_prompt = patched_get_prompt
    WhisperModel._is_patched = True

# --- EXÉCUTION ---
# 1. Transcription
model = whisperx.load_model("base", device, compute_type=compute_type)
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)

# 2. Alignement
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device)

# 3. Diarization
from whisperx.diarize import DiarizationPipeline
diarize_model = DiarizationPipeline(use_auth_token=YOUR_HF_TOKEN, device=device)
diarize_segments = diarize_model(audio)
result = whisperx.assign_word_speakers(diarize_segments, result)

# 4. Affichage
for segment in result["segments"]:
    print(f"[{segment['start']:.2f}s - {segment['end']:.2f}s] {segment.get('speaker', 'Inconnu')}: {segment['text']}")